# 12. Temperature: меняем выбор токена, не веса

Небольшой урок: **A — три числа, B — одна строка в генерации, C — сравнение текстов**. Начни только с A. Новых слоёв и новых оптимизаторов нет.

Ты сам напишешь функцию вероятностей и подключишь её к знакомому циклу. Готовый код прошлых уроков ниже нужен только для подготовки модели; переписывать его не требуется.

Ноутбук самостоятельный, старые ноутбуки запускать не нужно. Часть A требует только PyTorch. Для B–C используются локальный корпус урока 09 и однократное обучение на 300 шагах: урок 11 не сохранял checkpoint. Это новый подготовительный запуск, а не загрузка прежних весов. Между разными temperature обучения НЕ будет.

Ментор не выполнял ячейки. Сначала предположи результат, потом запусти свои решения и сохрани outputs. Коммит отдельно после разбора.

## A. Что именно меняет temperature

Модель выдала logits: `[2, 1, 0]`. Это ещё не вероятности. Обычно мы применяем softmax; теперь перед ним делим logits на положительное число `temperature`:

```text
scaled_logits = logits / temperature
probabilities = softmax(scaled_logits)
```

При temperature = 0.5 получим `[4, 2, 0]`: различия увеличились. При temperature = 2 получим `[1, 0.5, 0]`: различия уменьшились. Softmax преобразует эти различия в вероятности.

- `temperature = 1`: исходное распределение.
- `0 < temperature < 1`: распределение сильнее сосредоточено на самых больших logits.
- `temperature > 1`: распределение более равномерное.

Это не добавление случайного шума к весам. Порядок logits при положительной temperature не меняется: самый вероятный токен остаётся тем же, меняются вероятности его выбора и выбора альтернатив. Случайность появляется при multinomial.

В этой функции запрещаем 0 и отрицательные значения. Делить на ноль нельзя. Жадный выбор через argmax — отдельный способ; очень маленькая положительная temperature лишь приближает его при единственном максимуме.

**Твоё задание:** напиши одну строку return: раздели logits на temperature и примени softmax по последней оси. Делим именно logits, не готовые вероятности. Форма входа и выхода одинакова.

In [1]:
import math
import torch

def probabilities_with_temperature(logits, temperature):
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError('temperature должна быть конечной и положительной')
    return torch.softmax(logits / temperature, dim=-1)

In [2]:
# Три искусственных токена; это не результаты нашей языковой модели.
toy_logits = torch.tensor([[2.0, 1.0, 0.0]])
print('temperature | P(a)   P(b)   P(c)')
for temperature in [0.5, 1.0, 2.0]:
    probs = probabilities_with_temperature(toy_logits, temperature)
    assert probs.shape == (1, 3)
    assert torch.allclose(probs.sum(dim=-1), torch.ones(1))
    print(temperature, [round(p, 4) for p in probs[0].tolist()])

assert torch.allclose(probabilities_with_temperature(toy_logits, 1.0),
                      torch.softmax(toy_logits, dim=-1))
assert (probabilities_with_temperature(toy_logits, 0.5)[0, 0]
        > probabilities_with_temperature(toy_logits, 2.0)[0, 0])

temperature | P(a)   P(b)   P(c)
0.5 [0.8668, 0.1173, 0.0159]
1.0 [0.6652, 0.2447, 0.09]
2.0 [0.5065, 0.3072, 0.1863]


### Мои ответы — A

1. Запиши вероятность токена «a» при трёх temperature. Где она наибольшая?
2. Меняется ли самый вероятный токен? Почему?
3. Если веса модели не менялись, что именно изменилось?

**Остановка:** пришли «Сделал A». Для первой части не нужно запускать оставшиеся ячейки.

Ответы:
Меняется различие вероятности, самый вероятностный токен будет тот же.

### Разбор ментора

Исходный ответ сохранён. Проверка по коду и сохранённым outputs, без повторного запуска.

Код probabilities_with_temperature верен: деление logits на положительную temperature выполняется до softmax по последней оси.

1. В сохранённом примере P(a): при 0.5 — 0.8668, при 1.0 — 0.6652, при 2.0 — 0.5065. Наибольшая вероятность при 0.5.
2. Ты верно заметил: для этих же исходных logits самый вероятный токен остаётся тем же. Деление на положительное число и softmax сохраняют порядок значений. Это относится к одному и тому же контексту; после разных выбранных токенов контексты и следующие logits могут различаться.
3. Изменилось распределение вероятностей выбора токена, не веса модели. При меньшей temperature распределение более сосредоточено, при большей — более равномерно. Самый вероятный токен и случайно выбранный через multinomial токен — не обязательно один и тот же.

## Подготовка модели — готово, не переписываем

Ниже копия твоих функций и архитектуры из урока 11 (код можно развернуть в редакторе). Train — главы I–X «Капитанской дочки»; словарь только по train. Источник: `data/captains_daughter/README.md`. Test не оцениваем. Параметры прежние: D=32, H=4, 2 блока, окно 32.

Запусти определения и обучение один раз перед B. При повторном запуске ячейки обучения создаётся новая модель с нуля. Для сравнения temperature эту ячейку повторять не нужно.

In [3]:
import json
import math
import unicodedata
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

SEED = 42
CONTEXT_LENGTH = 32
BATCH_SIZE = 16
TRAIN_STEPS = 300
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)

candidates = [p for base in [Path.cwd(), *Path.cwd().parents]
              for p in [base, base / 'Seminars' / 'transformers']]
LESSON_DIR = next((p for p in candidates if (p / 'data/captains_daughter/chapters.json').is_file()), None)
if LESSON_DIR is None:
    raise FileNotFoundError('Открой ноутбук внутри проекта deepschool с локальным корпусом урока 09')
chapters = json.loads((LESSON_DIR / 'data/captains_daughter/chapters.json').read_text(encoding='utf-8'))
def normalize(text):
    return unicodedata.normalize('NFC', text).lower()
train_text = normalize('\n\n'.join(c['text'] for c in chapters[:10]))
val_text = normalize('\n\n'.join(c['text'] for c in chapters[10:12]))
tokens = ['<unk>'] + sorted(set(train_text))
stoi = {token: i for i, token in enumerate(tokens)}
vocab_size = len(tokens)
def encode(text):
    return torch.tensor([stoi.get(c, 0) for c in normalize(text)], dtype=torch.long)
def decode(ids):
    return ''.join(tokens[i] for i in ids.tolist())
train_ids, val_ids = encode(train_text), encode(val_text)
print('device:', DEVICE, '| V:', vocab_size)
print('train / val characters:', len(train_ids), len(val_ids))

device: cuda | V: 75
train / val characters: 130505 29122


In [4]:
def make_example(data, start, context_length):
    x = data[start:start+context_length]
    y = data[start+1:start+context_length+1]
    return x,y

In [5]:
def make_batch(data, starts, context_length):

    xs, ys = list(), list()
    for start in starts:
        x, y = make_example(data, start, context_length)
        xs.append(x)
        ys.append(y)
    xb = torch.stack(xs, dim=0).to(DEVICE)
    yb = torch.stack(ys, dim=0).to(DEVICE)

    return xb, yb

In [6]:
def sample_batch(data, generator):
    # high исключается: последний допустимый start = len(data) - T - 1.
    starts = torch.randint(len(data) - CONTEXT_LENGTH, (BATCH_SIZE,), generator=generator)
    xb, yb = make_batch(data, starts.tolist(), CONTEXT_LENGTH)
    return xb.to(DEVICE), yb.to(DEVICE)

In [8]:
class VectorizedCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError('Нужны положительные D,H и D, кратное H')
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def project_heads(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model or x.shape[1] == 0:
            raise ValueError('Ожидается непустой x формы (B,T,D)')
        batch_size, length, _ = x.shape
        # Готовая реализация из урока 05c: проекция и разделение голов.
        q, k, v = [
            projection(x)
            .reshape(batch_size, length, self.num_heads, self.head_dim)
            .transpose(1, 2)
            for projection in (
                self.query_projection,
                self.key_projection,
                self.value_projection,
            )
        ]

        return q, k, v

    def forward(self, x):
        batch_size, length, _ = x.shape
        q, k, v = self.project_heads(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        masked_scores = scores.masked_fill(blocked, float('-inf'))
        weights = torch.softmax(masked_scores, dim=-1)
        context = weights @ v
        combined = context.transpose(1, 2).reshape(batch_size, length, self.d_model)
        return self.output_projection(combined)

In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.norm_1 = nn.LayerNorm(d_model)
        self.attention = VectorizedCausalAttention(d_model, num_heads)
        self.norm_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        x = x + self.attention(self.norm_1(x))
        x = x + self.mlp(self.norm_2(x))
        return x

In [10]:
class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len=16, d_model=8,
                 num_heads=2, d_ff=32, n_layers=1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        if token_ids.ndim != 2:
            raise ValueError('Ожидаются token IDs формы (B,T)')
        length = token_ids.shape[1]
        if not 1 <= length <= self.max_seq_len:
            raise ValueError('Длина должна быть от 1 до max_seq_len')
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

In [11]:
def new_model():
    return TinyLanguageModel(vocab_size=vocab_size, max_seq_len=CONTEXT_LENGTH,
                             d_model=32, num_heads=4, d_ff=128, n_layers=2).to(DEVICE)

In [12]:
def language_model_loss(logits, targets):
    V = logits.shape[-1]
    logits = logits.reshape(-1, V)
    targets = targets.reshape(-1)
    loss = F.cross_entropy(logits, targets)
    return loss

In [13]:
def train_step(model, optimizer, xb, yb):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    logits = model(xb)

    loss = language_model_loss(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item()

In [14]:
torch.manual_seed(SEED)
model = new_model()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_rng = torch.Generator().manual_seed(43)
for step in range(1, TRAIN_STEPS + 1):
    xb, yb = sample_batch(train_ids, train_rng)
    batch_loss = train_step(model, optimizer, xb, yb)
    if step % 100 == 0:
        print('Подготовка, шаг:', step, '| batch loss:', round(batch_loss, 4))
model.eval()
# Снимок для проверки: последующая генерация не должна менять состояние модели.
before_generation = {name: value.detach().clone() for name, value in model.state_dict().items()}

Подготовка, шаг: 100 | batch loss: 2.9476
Подготовка, шаг: 200 | batch loss: 2.6709
Подготовка, шаг: 300 | batch loss: 2.7032


## B. Подключаем temperature к генерации

Остальной цикл уже знаком. Вместо прежнего softmax вызови свою функцию из A, передав ей last_logits и temperature.

Формы: logits `(B,T,V)` → last_logits `(B,V)` → probs `(B,V)` → next_token `(B,1)`. Temperature не меняет формы. `num_samples=1` по-прежнему выбирает один следующий токен для каждой последовательности.

Не меняй веса, длину окна или число шагов обучения. Наша задача — менять только способ преобразования logits перед выбором токена.

In [16]:
@torch.no_grad()
def generate_with_temperature(model, idx, max_new_tokens, context_length, temperature=1.0):
    model.eval()
    for _ in range(max_new_tokens):
        logits = model(idx[:, -context_length:])
        last_logits = logits[:, -1, :]
        probs = probabilities_with_temperature(last_logits, temperature)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return idx

## C. Один управляемый эксперимент

**Перед запуском запиши гипотезу ниже.** Сравним temperature 0.5, 1.0 и 1.5. Веса, промпт и длина продолжения одинаковы. Для каждого значения используем одни и те же два seed: не выбираем только удачный пример.

Одинаковый seed не гарантирует одинаковый текст при разных temperature: вероятности меняются, а после различающихся токенов меняется и контекст. В одной среде seed позволяет воспроизводить выбор; между версиями PyTorch и устройствами точное совпадение не обещаем.

Всего будет шесть коротких продолжений. Сравни повторы, похожие на слова фрагменты и связность. Два seed — иллюстрация, не доказательство, что какая-то temperature всегда лучше. Не путай переданный промпт «отец мой» с текстом, который сгенерировала модель.

### Моя гипотеза до запуска

При temperature = 0.5 ожидаю более часто повторяемые последовательности токенов

При temperature = 1.5 ожидаю разброс последовательностей токенов. будет меньше повторов второй буквы после определнной первой

In [17]:
prompt = encode('отец мой ').unsqueeze(0).to(DEVICE)
for temperature in [0.5, 1.0, 1.5]:
    for seed in [2026, 2027]:
        torch.manual_seed(seed)
        result = generate_with_temperature(model, prompt.clone(), 80,
                                           CONTEXT_LENGTH, temperature)
        assert result.shape == (1, prompt.shape[1] + 80)
        assert torch.equal(result[:, :prompt.shape[1]], prompt)
        print(f'temperature={temperature}, seed={seed}')
        print(decode(result[0].cpu()))
        print()

assert all(torch.equal(before_generation[name], value)
           for name, value in model.state_dict().items())
print('Состояние модели не изменилось.')

temperature=0.5, seed=2026
отец мой наво я м о нале рету сто, я вала восво о чеданадалезале ма сл кареся вовочелатоб

temperature=0.5, seed=2027
отец мой виторегос станеме поданоза ить по но ностеронинома о водорол мего воводри си вов

temperature=1.0, seed=2026
отец мой н вроя м оствилорету сис, я валь восвово ч:дчндду, забе у.асл карыелавоврчелрист

temperature=1.0, seed=2027
отец мой витатя. с ссмнема педанозадитьй бьго, нсведоебнре иричия мо. мего зо— дри си воа

temperature=1.5, seed=2026
отец мой о врця м ос“вилор ту сис, я валь воtцово ч:ьчндду…гззбе у.асл карpсямргвзчюлрися

temperature=1.5, seed=2027
отец мой эбуканори есмнщере! ьхноза*!тьйши.г енысв!ê пбнре иричия*ро. мего зе— дри сh чта

Состояние модели не изменилось.


### Мои ответы — C

1. Подтвердилась ли гипотеза? Приведи по короткому примеру для 0.5 и 1.5, обозначив seed.
2. Стала ли меньшая temperature гарантией осмысленного текста? Что наблюдаешь с повторами?
3. Закончи фразу: «Temperature меняет …, но не …».

Ответы:
При низкой температуре больше осмысленных слогов (двух-трех символов)
Меняет различие вероятностей, но не перемешивает их порядок в токенах

### Разбор ментора

Исходные ответы и гипотеза сохранены.

1. Твоё наблюдение разумно, но лучше говорить «похожие на русские буквосочетания», а не «осмысленные слоги»: смысл и правильное слогоделение мы не проверяли. Например, при temperature=0.5, seed=2027 встречается «по но ностеронинома», а при 1.5 с тем же seed — «ьхноза*!тьйши». При 1.5 появились более необычные сочетания и символы, в том числе ê. Это токены имеющегося словаря, а не новые символы, созданные temperature. Промпт «отец мой» не считаем результатом генерации.
2. Низкая temperature не дала связного текста. В выводе 0.5, seed=2027 видны повторяющиеся фрагменты «во», например в «воводри си вов». Однако число повторов мы не считали: два коротких примера не доказывают, что при 0.5 их всегда больше, а при 1.5 всегда меньше. Гипотеза о более разнообразном выборе согласуется с изменением распределения; гипотеза об уменьшении повторов требует отдельной проверки. Модель учитывает доступный контекст, а не только одну предыдущую букву.
3. Полная формулировка: «Temperature меняет распределение вероятностей выбора следующего токена, но не веса модели и не её знания». Для фиксированных logits при положительной temperature сохраняется ранжирование токенов по вероятности. Это НЕ означает сохранение порядка токенов в сгенерированном тексте: сам выбор случаен, а после разных выборов различается дальнейший контекст.

Код подключения к генерации верен. В сохранённом выводе проверка сообщает, что состояние модели не изменилось. Части A–C разобраны; урок завершён по коду, ответам и сохранённым результатам, без нового запуска ментора.

## Что забрать из урока

Temperature меняет распределение выбора следующего токена. Она не обучает модель, не увеличивает окно и не добавляет знаний. Маленькая temperature может усиливать повторы; большая может повышать разнообразие вместе с числом неудачных вариантов. Это не обещание о каждом отдельном тексте.

Мы не измеряли улучшение стандартного validation loss и не можем его объявить по генерациям: forward модели не менялся. Если отдельно вычислять cross-entropy по logits / temperature, это уже другая оценка распределения, которой здесь не занимались.

Для собеседования: объясни порядок «logits → деление на temperature → softmax → multinomial» и отличие от argmax. После разбора перейдём к checkpoint, чтобы больше не обучать модель заново ради каждого короткого эксперимента.